# CP-SAT 简介 ― 用 SAT 技术做约束编程

## 本章要点

OR-Tools 的 **CP-SAT** 是 Google 开发的约束编程求解器。它和 Z3 的章节一样，都是“写出条件、让程序求解”的工具，但更擅长**最优化**（让利润最大、让距离最小）。

- CP-SAT 靠什么机制求解（SAT 的技术＋线性松弛）
- 最基本的用法（`CpModel` / `new_int_var` / `add` / `solve`）
- 线性规划的例题（最大化利润）
- 用 `assumptions` 推理逻辑谜题

## CP-SAT 是什么

CP-SAT 是**以 SAT（可满足性问题）技术为基础的约束编程求解器**。

- 它用 SAT 求解器的 **CDCL**（冲突学习并剪枝）和 **LCG**（Lazy Clause Generation，惰性子句生成）等技术，聪明地推进搜索。
- 它还组合了**线性松弛**，所以能像 MIP（混合整数规划）求解器那样，做“让目标函数最大／最小”的最优化。

OR-Tools 里早就有 `constraint_solver`（经典 CP 求解器），但它以深度优先搜索为主，不擅长最优化。**新用 OR-Tools 就选 CP-SAT**，这是定论。本书的 OrTools 章节全部使用 CP-SAT。

(ortools-send-more-money)=
## 例题：SEND + MORE = MONEY

[用 Z3 求解 SEND+MORE=MONEY](z3-send-more-money)

Z3 的章节里也做过的字母算术，这次用 CP-SAT 来解。用法的流程，全都写在这段代码里。

In [1]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()
s, e, n, d, m, o, r, y = [model.new_int_var(0, 10, name) for name in "sendmory"]
model.add(s != 0)
model.add(m != 0)
model.add_all_different([s, e, n, d, m, o, r, y])
send = s * 1000 + e * 100 + n * 10 + d
more = m * 1000 + o * 100 + r * 10 + e
money = m * 10000 + o * 1000 + n * 100 + e * 10 + y

model.add(send + more == money)
solver = cp_model.CpSolver()
solver.Solve(model)
print(f"{solver.value(send)} + {solver.value(more)} = {solver.value(money)}")

9567 + 1085 = 10652


### 代码说明

- `CpModel()` … 创建模型（装约束的容器）
- `model.new_int_var(下限, 上限, 名字)` … 创建整数变量
- `model.add(条件)` … 添加约束（`!=`, `==`, `>=` 等比较式可以直接写）
- `model.add_all_different([...])` … 让所有值互不相同
- `solver.solve(model)` … 求解
- `solver.value(式)` … 取出式子的值

用 `*` 和 `+` 就能拼出式子，所以像 `s * 1000 + e * 100 + ...` 这样把“数位”直接写出来。加权和也可以用 `LinearExpr.weighted_sum()` 写。

```python
send = cp_model.LinearExpr.weighted_sum([s, e, n, d], [1000, 100, 10, 1])
more = cp_model.LinearExpr.weighted_sum([m, o, r, e], [1000, 100, 10, 1])
money = cp_model.LinearExpr.weighted_sum([m, o, n, e, y], [10000, 1000, 100, 10, 1])
```

## 线性规划

接下来是“让利润最大”的问题。作为线性规划的例子，考虑下面的问题。

生产产品 A 和 B 需要三种原料 a, b, c。生产 1 单位所需的量和库存如下，卖出 1 单位，A 有 3 万日元利润，B 有 2 万日元利润。请决定使利润最大的生产量。

| 原料 | 产品 A | 产品 B | 库存 |
|------|--------|--------|------|
| a    | 3      | 1      | 9    |
| b    | 2.5    | 2      | 12.5 |
| c    | 1      | 2      | 8    |

设生产量为 $x_1$, $x_2$，问题可以这样写。

**目标函数**

$$
z = 3x_1 + 2x_2 \quad \text{（最大化）}
$$

**约束条件**

$$
\begin{aligned}
3x_1 + x_2 &\leq 9 \\
2.5x_1 + 2x_2 &\leq 12.5 \\
x_1 + 2x_2 &\leq 8 \\
x_1, x_2 &\geq 0
\end{aligned}
$$

CP-SAT 只能处理**整数**。含有 `2.5` 的式子，把两边都乘以 2 变成整数（`2.5x1 + 2x2 <= 12.5` → `5x1 + 4x2 <= 25`）。不等式两边同乘 2，条件的意思不变。

`model.maximize(z)` 的意思是“寻找让 $z = 3x_1 + 2x_2$ 最大的 $x_1, x_2$”。

In [2]:
model = cp_model.CpModel()
x1, x2 = [model.new_int_var(0, 10000, f"x{i}") for i in range(1, 3)]
model.add(3 * x1 + x2 <= 9)
model.add(5 * x1 + 4 * x2 <= 25)
model.add(x1 + 2 * x2 <= 8)
z = 3 * x1 + 2 * x2
model.maximize(z)
solver = cp_model.CpSolver()
solver.Solve(model)
print(f"x1 = {solver.value(x1)}, x2 = {solver.value(x2)}, z = {solver.value(z)}")

x1 = 2, x2 = 3, z = 12


## 逻辑谜题

有红、蓝、黄、绿四个箱子。其中几个箱子里放着苹果（至少 1 个，2 个以上也可以）。

你可以选一个箱子，拿走里面的东西。请根据下面的信息，选出**一定放着苹果的箱子**。

- 如果红箱子里有苹果，那么蓝箱子里也有苹果。
- 如果绿箱子里没有苹果，那么黄箱子和蓝箱子里都没有苹果。

在 CP-SAT 里，用布尔变量和逻辑约束就能直接写出来。

In [3]:
model = cp_model.CpModel()
red, blue, yellow, green = [model.new_bool_var(name) for name in ['red', 'blue', 'yellow', 'green']]

model.add_bool_or([red, blue, yellow, green])                 # 至少有 1 个
model.add_implication(red, blue)                               # 红 → 蓝
model.add_bool_and(~yellow, ~blue).only_enforce_if(~green)     # 没有绿 → 黄和蓝也没有

solver = cp_model.CpSolver()
for box in [red, blue, yellow, green]:
    model.add_assumption(~box)          # 假设“这个箱子里没有”
    res = solver.solve(model)
    if res == cp_model.INFEASIBLE:      # 矛盾 → 一定放着
        print(box)
    model.clear_assumptions()           # 删掉假设

green


### 代码说明

- `new_bool_var()` … 创建 0/1 的布尔变量
- `add_bool_or([...])` … 至少 1 个为真（逻辑或）
- `add_implication(a, b)` … 若 `a` 则 `b`（逻辑蕴含）
- `add_bool_and(a, b).only_enforce_if(c)` … 只在 `c` 成立时“`a` 且 `b`”
  - `~` 是取反
  - `only_enforce_if()` 指定“只在这个条件成立时有效”

这个谜题的目的不是找一个解，而是推理**哪个箱子一定放着苹果**。于是对每个箱子，假设“里面没有”再求解，**如果矛盾（`INFEASIBLE`），就说明那个箱子里一定放着苹果**。

- 用 `model.add_assumption(条件)` 添加假设
- 用 `clear_assumptions()` 删除假设（试下一个箱子前必须调用）

结果是 `green`。因为如果没有绿，黄和蓝也没有，红→蓝也用不上，于是一个苹果都不剩。

## 小结

- CP-SAT 是靠 SAT 技术＋线性松弛工作的约束求解器。用 OR-Tools 就选它
- 基本流程是 `CpModel()` → 变量 → `add()` → `solve()` → `value()`
- 最优化用 `model.maximize(式)` / `model.minimize(式)`
- CP-SAT 只处理整数。带小数的式子，把两边同乘整数消掉
- 用布尔变量和逻辑约束（`add_bool_or`, `add_implication`, `only_enforce_if`）也能写条件分支
- 用 `assumptions` 可以检查“加上这个条件会不会矛盾”（推理谜题的定式）

下一章，我们更详细地看看 CP-SAT 的 API。